In [3]:
import requests
import json
import time

url = 'https://d5dg1j9kt695d30blp03.apigw.yandexcloud.net'
nickname = "testuser"
cohort = 26



headers = {
    "X-API-KEY": "5f55e6c0-e9e5-4a9c-b313-63c01fc31460",
    "X-Nickname": nickname,
    "X-Cohort": str(cohort)

}

method_url = '/generate_report'

r = requests.post(url + method_url, headers=headers)

response_dict = json.loads(r.content)

print(response_dict)
task_id = response_dict['task_id']
time.sleep(130)

method_url = '/get_report'
payload = {'task_id': task_id}

r = requests.get(url + method_url, params=payload, headers=headers)

response_dict = json.loads(r.content)

print(response_dict)

report_id = response_dict['data']['report_id']

#Качаем сами файлы - все работает!
import datetime 
print(datetime.datetime.now())
import pandas as pd
files = ('customer_research.csv', 'user_order_log.csv', 'user_activity_log.csv', 'price_log.csv')

for file in files:
    url = f"https://storage.yandexcloud.net/s3-sprint3/cohort_{cohort}/{nickname}/{report_id}/{file}"
    print(url)
    pd.read_csv(url).to_csv(f'./{file}', index = False)
    print(f'Downloaded {file}')

{'task_id': 'MjAyNC0wNS0yM1QxNjozMzoxNwl4MzEyOTYxMg=='}
{'task_id': 'MjAyNC0wNS0yM1QxNjozMzoxNwl4MzEyOTYxMg==', 'status': 'SUCCESS', 'data': {'report_id': 'TWpBeU5DMHdOUzB5TTFReE5qb3pNem94TndsNE16RXlPVFl4TWc9PQ==', 'start_day': '2024-04-23 00:00:00', 'end_date': '2024-05-15 00:00:00', 's3_path': {'customer_research': 'https://storage.yandexcloud.net/s3-sprint3/cohort_26/x3129612/TWpBeU5DMHdOUzB5TTFReE5qb3pNem94TndsNE16RXlPVFl4TWc9PQ==/customer_research.csv', 'user_order_log': 'https://storage.yandexcloud.net/s3-sprint3/cohort_26/x3129612/TWpBeU5DMHdOUzB5TTFReE5qb3pNem94TndsNE16RXlPVFl4TWc9PQ==/user_order_log.csv', 'user_activity_log': 'https://storage.yandexcloud.net/s3-sprint3/cohort_26/x3129612/TWpBeU5DMHdOUzB5TTFReE5qb3pNem94TndsNE16RXlPVFl4TWc9PQ==/user_activity_log.csv', 'price_log': 'https://storage.yandexcloud.net/s3-sprint3/cohort_26/x3129612/TWpBeU5DMHdOUzB5TTFReE5qb3pNem94TndsNE16RXlPVFl4TWc9PQ==/price_log.csv'}}}
2024-05-23 19:35:31.614537
https://storage.yandexcloud.net/s3-

In [1]:
from pyspark.sql import SparkSession
from langchain.llms import OpenAI
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

# Инициализация Spark сессии
spark = SparkSession.builder \
    .appName("LangChainSparkIntegration") \
    .getOrCreate()

# Пример данных
data = [("Что такое искусственный интеллект?",), 
        ("Объясни теорию относительности",),
        ("Как работает блокчейн?",)]
df = spark.createDataFrame(data, ["question"])

# Инициализация LangChain компонентов
def init_langchain():
    llm = OpenAI(temperature=0.7, model_name="gpt-3.5-turbo")
    
    prompt_template = PromptTemplate(
        input_variables=["question"],
        template="Ответь на следующий вопрос: {question}"
    )
    
    return LLMChain(llm=llm, prompt=prompt_template)

# Функция для обработки вопросов с помощью LangChain
def process_question(question):
    # Важно: инициализация для каждого рабочего процесса
    chain = init_langchain()
    return chain.run(question=question)

# Регистрация UDF
process_question_udf = udf(process_question, StringType())

# Применение LangChain к данным Spark
result_df = df.withColumn("answer", process_question_udf(df["question"]))

# Показать результаты
result_df.show(truncate=False)

# Остановка Spark сессии
spark.stop()

ModuleNotFoundError: No module named 'langchain_community'